### Compare the initial and the current state of the design file: 
- are sequences missing? 
- At which point got they missing? 

In [2]:
import ast
from importlib import reload
import numpy as np
import os
import pandas as pd
import re
import yaml


# load helpful functions
import sys
sys.path.append('../../../00_helpful_functions')
import helpful_functions as hf
reload(hf)
# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

### Check header duplicates: 

In [170]:
design_file_new_deduplication_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences.fa'
design_file_new_deduplication_path = '/home/kisa/coding/80K_MPRA/design_data/design_info/design_removed_spaces_deduplicated_sequences_renamed.fa'
design_file_new_deduplication = hf.fasta_to_dataframe(design_file_new_deduplication_path)
design_file_new_deduplication['no_adapter'] = design_file_new_deduplication['sequence'].apply(lambda x: x[15:-15])

In [171]:
design_file_new_deduplication.loc[design_file_new_deduplication.duplicated(subset=['header'])]

,header,sequence,no_adapter


In [172]:
design_file_new_deduplication.loc[design_file_new_deduplication['header'].str.contains('GC_Mendelian_variants:REF_chr8:1170389')]['header'].to_list()
# design_file_new_deduplication.loc[design_file_new_deduplication['header'].str.contains('GC_Mendelian_variants:REF_chr8')]

['GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4']

In [173]:
design_file_new_deduplication.loc[(design_file_new_deduplication['header'].str.contains('GC_Mendelian_variants')) & (design_file_new_deduplication['header'].str.contains('ALT_')) & (design_file_new_deduplication['header'].str.contains('REF_'))]['header'].to_list()
# ['GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4#GC_Mendelian_variants:ALT_chr8:11703860G>T|GATA4_chr8:11703890AG>A|GATA4',
#  'GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4#GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4']



[]

In [174]:
design_file_new_deduplication.loc[(design_file_new_deduplication['header'].str.contains('#')) & (design_file_new_deduplication['header'].str.contains('ALT_')) & (design_file_new_deduplication['header'].str.contains('REF_'))]['header'].to_list()


['GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3#GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT']

In [175]:
design_file_new_deduplication_duplicates = design_file_new_deduplication.loc[design_file_new_deduplication['header'].str.contains('#')].copy()
# get label of the sequence and the label of the sequence after the # => find out which groups need more attention
design_file_new_deduplication_duplicates['label'] = design_file_new_deduplication_duplicates['header'].apply(hf.get_label)

design_file_new_deduplication_duplicates['second_label'] = design_file_new_deduplication_duplicates['header'].apply(lambda header: hf.get_label(header.split('#')[1]))
design_file_new_deduplication_duplicates['third_label'] = design_file_new_deduplication_duplicates['header'].apply(lambda header: hf.get_label(header.split('#')[2]) if len(header.split('#')) > 2  else 'NA')
design_file_new_deduplication_duplicates['fourth_label'] = design_file_new_deduplication_duplicates['header'].apply(lambda header: hf.get_label(header.split('#')[3]) if len(header.split('#')) > 3  else 'NA')
design_file_new_deduplication_duplicates['n_duplicates'] = design_file_new_deduplication_duplicates['header'].apply(lambda header: len(header.split('#')))

In [176]:
design_file_new_deduplication_duplicates.n_duplicates.value_counts()

n_duplicates
3    245
2     96
4      1
Name: count, dtype: int64

In [177]:
design_file_new_deduplication_duplicates['second_label'].value_counts()


second_label
MK                       263
GC_Glut_Chengyu           43
C_negative_neuron_MK      12
GC_Mendelian_variants     12
GC_Mohlke                  5
C_positive_neuron_MK       3
C_positive_heart_CAD       2
C_positive_neuron_CD       2
Name: count, dtype: int64

In [178]:
design_file_new_deduplication_duplicates['third_label'].value_counts()


third_label
MK                      245
NA                       96
C_positive_neuron_MK      1
Name: count, dtype: int64

In [179]:
design_file_new_deduplication_duplicates['fourth_label'].value_counts()

fourth_label
NA    341
MK      1
Name: count, dtype: int64

In [180]:
design_file_new_deduplication_duplicates['label'].value_counts()

label
MK                       242
GC_GABA_Chengyu           46
C_negative_heart_MK       12
GC_Mendelian_variants     12
C_positive_neuron_MK      10
GC_Selvarajan              6
C_positive_heart_MK        6
C_negative_neuron_MK       6
GC_Mohlke                  1
C_positive_neuron_CD       1
Name: count, dtype: int64

In [181]:
# interested in groups with duplicates: union and set
set(design_file_new_deduplication_duplicates['label'].to_list()).union(set(design_file_new_deduplication_duplicates['second_label'].to_list())).union(set(design_file_new_deduplication_duplicates['third_label'].to_list())).union(set(design_file_new_deduplication_duplicates['fourth_label'].to_list()))

{'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_positive_heart_CAD',
 'C_positive_heart_MK',
 'C_positive_neuron_CD',
 'C_positive_neuron_MK',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Mendelian_variants',
 'GC_Mohlke',
 'GC_Selvarajan',
 'MK',
 'NA'}

In [ ]:
design_file_new_deduplication_duplicates

In [24]:
mendelian_variant_map_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/final_design/results/final_design/GC_Mendelian_variants/variant_region_map.tsv.gz'
mendelian_variant_map = pd.read_csv(mendelian_variant_map_path, sep="\t")
mendelian_variant_map.columns = ['ID', 'Region', 'REF', 'ALT']

In [33]:
mendelian_variant_map.loc[(mendelian_variant_map['REF'].str.contains('GC_Mendelian_variants:REF_chr8\:11703890'))]


,ID,Region,REF,ALT
166,GC_Mendelian_variants:chr8:11703860G>T|GATA4,GC_Mendelian_variants:chr8:11703890AG>A|GATA4,GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4,GC_Mendelian_variants:ALT_chr8:11703890AG>A|GA...
168,GC_Mendelian_variants:chr8:11703890AG>A|GATA4,GC_Mendelian_variants:chr8:11703890AG>A|GATA4,GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4,GC_Mendelian_variants:ALT_chr8:11703890AG>A|GA...


In [ ]:
# add code for the manual change of the headers to the preprocessing on the server

### Investigatet the following sequences in more detail: headers which were duplicated
- for GC_Mendelian_variants:REF_chr8:11703890A: two times reference: ref1 is similar to alt1
  - 

In [ ]:
# GC_Mendelian_variants:REF_chr8:11703890A
# GC_Mendelian_variants:REF_chr8:11703860G
# GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3

In [15]:
print('number of sequences which were duplicated: ')
design_file_new_deduplication.loc[(design_file_new_deduplication['header'].str.contains('#'))].shape[0] # 344 duplicated sequences


,header,sequence
74642,GC_Mendelian_variants:REF_chr8:11703890AG>A|GA...,AGGACCGGATCAACTCCAGGAACTAGCATCCAGCCGGGCACCCCGG...


In [38]:
print(design_file_new_deduplication.loc[(design_file_new_deduplication['header'].str.contains('GC_Mendelian_variants:REF_chr8:11703890A'))]['header'].to_list())
# get sequence of the ref + alt match
alt_sequence_list = design_file_new_deduplication.loc[design_file_new_deduplication['header'] == 'GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4#GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4']['sequence'].to_list()
ref_sequence_list = design_file_new_deduplication.loc[design_file_new_deduplication['header'] == 'GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4']['sequence'].to_list()
# 11703890:
# 11703755 - 11704024


['GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4#GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4', 'GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4']


In [40]:
[seq[15:-15] for seq in ref_sequence_list]
# [seq[15:-15] for seq in alt_sequence_list]

['CCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGG']

Dieser header macht vorne und hinten keinen Sinn
- https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr8%3A11703868%2D11703933&hgsid=2390860399_gGWDS3zXLR4BG0VkSs24Se3k59mG
- die sequenz mit dem namen ref hat keine 100% matching auf dem human genome
- die sequenz mit dem alt hat 100% match mit dem human genome hg38

In [ ]:
ref:CCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGA
    ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
alt:CCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGG

In [34]:
blat_start = 11703755
blat_end = 11704024
var_pos = 11703890

#### Next one from the mendelian variants
- again the alt_sequence has 100% matches on chr8 + strand 11703725  11703994
- the ref sequence has one gap

In [52]:
print(design_file_new_deduplication.loc[(design_file_new_deduplication['header'].str.contains('GC_Mendelian_variants:REF_chr8:11703860G'))]['header'].to_list())
# # get sequence of the ref + alt match
# alt_sequence_list = design_file_new_deduplication.loc[design_file_new_deduplication['header'] == 'GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4#GC_Mendelian_variants:ALT_chr8:11703860G>T|GATA4_chr8:11703890AG>A|GATA4']['sequence'].to_list()
# ref_sequence_list = design_file_new_deduplication.loc[design_file_new_deduplication['header'] == 'GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4']['sequence'].to_list()

['GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4#GC_Mendelian_variants:ALT_chr8:11703860G>T|GATA4_chr8:11703890AG>A|GATA4', 'GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4']


In [50]:
[seq[15:-15] for seq in ref_sequence_list]
# [seq[15:-15] for seq in alt_sequence_list] #

['CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCC']

In [ ]:
ref:CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCC
alt:CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCC

### Rename these sequences

In [169]:
config['region_bed']

KeyError: 'region_bed'

In [ ]:
design_file_new_deduplication

#### Check mohlke: 
- GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3
- chr1   +   230159137 230159406 (270) (alt)
- chr1   +   230159137 230159427 (269) (span 291) (ref)

In [182]:
design_file_new_deduplication.loc[design_file_new_deduplication['header'].str.contains('GC_Mohlke:REF_NC000001.11\|230158967\|C\|A\|MohlkeHepControls,NC000001.11\|230159168\|C\|T\|MohlkeHepControls,NC000001.11\|230159329\|CTTAAAGTGTTCAGCACTCCCCT\|CT\|MohlkeHepControls_fwd_tile3-3')]['header'].to_list()
# design_file_new_deduplication.loc[design_file_new_deduplication['header'] == 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3']['no_adapter'].to_list()
# design_file_new_deduplication.loc[design_file_new_deduplication['header'].str.contains('GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT', regex=False)]['no_adapter'].to_list()

['GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3#GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3']

In [ ]:
rename_map = {
    GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3#GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT
}

In [ ]:
rename_map = {
    "GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3": "GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT",
    "GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3#GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT": "GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3",
}

### After checking that for all these sequences the ref and the alt is changed we look for all reference sequences with bwa in the human genome
- conda create -n bwa bioconda::bwa
- !bedtools complement -i /home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/wtc11_ngn2_atac_peak_sorted.bed -g /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa.genome > /home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2.bed
- bwa mem -k 270 -T 30 /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa /home/kisa/coding/80K_MPRA/reference_genome_match/only_references_80K_design.fa > ref_alignment_hg38.sam
- bwa-mem2 -k 270 -T 30 -t 30 /home/kisa11/work/projects/static_data/hg38/hg38.fa /home/kisa11/work/projects/80K_MPRA/reference_genome_match/only_references_80K_design.fa > /home/kisa11/work/projects/80K_MPRA/reference_genome_match/ref_alignment_hg38.sam




In [118]:
only_references = design_file_new_deduplication.loc[design_file_new_deduplication['header'].str.contains(':REF_')]
only_references['no_adapter'] = only_references['sequence'].apply(lambda x: x[15:-15])
# add id because of query name too long error
only_references['header_length'] = only_references['header'].apply(lambda header: len(header))
def shorten_headers(row, threshold=1000, left_pos=57, right_pos=53):
    """Shortens headers if the header length is > threshold"""
    header = row['header']
    if len(header) > threshold:
        print(header[:left_pos] + header[-right_pos:])
        return header[:left_pos] + header[-right_pos:]
    return header
# only_references['new_header'] = only_references.apply(lambda row: shorten_headers(row, threshold=1000, left_pos=57, right_pos=53), axis=1)

# error persists: add short index
only_references = only_references.rename_axis('index1').reset_index()
only_references['index1'] = only_references['index1'].astype(str)
hf.write_fasta(only_references, '/home/kisa/coding/80K_MPRA/reference_genome_match/only_references_80K_design.fa', header=['index1', 'no_adapter'])

/tmp/ipykernel_489150/3067459617.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  only_references['no_adapter'] = only_references['sequence'].apply(lambda x: x[15:-15])
/tmp/ipykernel_489150/3067459617.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  only_references['header_length'] = only_references['header'].apply(lambda header: len(header))


True

In [105]:
reference_num = 6

only_references.loc[only_references['header_length'] > 200]['header'].to_list()[reference_num][:57] + only_references.loc[only_references['header_length'] > 200]['header'].to_list()[reference_num][-53:]

'GC_Kircher:REF_NC000001.11|109274794|C|T|KircherControls,NC000001.11|109275240|C|A|KircherControls_fwd_tile5-5'

In [85]:
pattern = 'GC_Mohlke:REF_NC000001.11|230158967|C'
only_references.loc[only_references['header'].str.contains(pattern, regex=False)]['header'].to_list()

['GC_Selvarajan:REF_rs4846913|STARR-seq-HepG2_fwd_tile1-1#GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile2-3',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3#GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|Mohlke

In [81]:
only_references['header_length'].describe()

only_references.boxplot(column=['header_length'])
only_references_sorted = only_references.sort_values(by='header_length').copy()
only_references_sorted.plot.bar(x='header', y='header_length', rot=90)

<Axes: xlabel='header'>

In [74]:
!cat /home/kisa/coding/80K_MPRA/reference_genome_match/only_references_80K_design.fa | grep ">" | wc -l # 18894 sequences

18894


computed on the charite cluster and copied back:
scp kisa11@s-sc-frontend3.charite.de:/home/kisa11/work/projects/80K_MPRA/reference_genome_match/perfect_matched_sequences.txt perfect_matched_sequences.txt

In [139]:
perfect_ref_match_path = '/home/kisa/coding/80K_MPRA/reference_genome_match/perfect_matched_sequences.txt'
perfect_ref_match_path = '/home/kisa/coding/80K_MPRA/reference_genome_match/modified_perfect_matched_sequences.txt'
perfect_matched_df_subset = pd.read_csv(perfect_ref_match_path, sep="\t", on_bad_lines='skip', header=None)
perfect_matched_df_subset = pd.read_csv(perfect_ref_match_path, sep="\t", header=None)
# perfect_ref_match_header_path = '/home/kisa/coding/80K_MPRA/reference_genome_match/perfect_matched_ids.txt'
# perfect_matched_header_df = pd.read_csv(perfect_ref_match_header_path, sep="\t", header=None)
perfect_matched_df_subset

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,8900,0,chr1,2179508,0,270M,*,0,0,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
1,8901,0,chr1,2191263,0,270M,*,0,0,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
2,8902,0,chr1,2191972,0,270M,*,0,0,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGAGGC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
3,8903,0,chr1,2192250,0,270M,*,0,0,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
4,8904,0,chr1,2192937,0,270M,*,0,0,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18886,27477,16,chrX,154531980,0,270M,*,0,0,TGGAGGTGGGTGCCCAGGGCTCAGAGCTTGTGGGGGTTCACCCACT...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
18887,27478,16,chrX,154539056,0,270M,*,0,0,AGTGCAGTAAGGCAAGAAAATGAAAATGGCTGAAGTTGAGCTGTGA...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
18888,27479,16,chrX,154545001,0,270M,*,0,0,CTTGAGTGCATCCAGGGTTAAAAGTGAAAACTGAGACCACGTGTAA...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN
18889,27480,16,chrX,154549803,0,270M,*,0,0,GTTGTAGACATCTTATATAAATAGACTCATACAATGTTTAATCTTT...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:0,NaN


In [146]:
# only unique ids:
perfect_matched_df_subset[perfect_matched_df_subset.columns.values[0]].nunique() # 18891 (one line per id)

18891

In [ ]:
matchable_header_subset = set(perfect_matched_df_subset[perfect_matched_df_subset.columns.values[0]].to_list())
# matchable_header = set(perfect_matched_header_df[perfect_matched_header_df.columns.values[0]].to_list())

In [145]:
perfect_matched_df_subset[perfect_matched_df_subset.columns.values[15]].notna().sum() # 243 have mulitple matches in the genome
perfect_matched_df_subset.loc[perfect_matched_df_subset[perfect_matched_df_subset.columns.values[15]].notna()]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
1396,10268,16,chr1,155295243,0,270M,*,0,0,GGGGTCCACAGTCACCAGCACCTGGGAGCCCTTCACCAGCTCCACT...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr1_GL383519v1_alt,-100266,270M,0;"
1397,10269,16,chr1,155295725,0,270M,*,0,0,CTTGAGGCGCTCCACGGAGCGAGATGCTGGCCCTAGAACCAGAGAT...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr1_GL383519v1_alt,-100748,270M,0;"
3303,12139,16,chr11,492774,0,270M,*,0,0,CTCTTTCCAGTCACGGCCACTCACAGCTTTGGCCTCCGTGGCCCTG...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr11_KI270832v1_alt,-22415,270M,0;"
3304,12140,16,chr11,494817,0,270M,*,0,0,TCCCCCACCCCCGCCTTCCTCCCTGGGCAGCCCTGGCCGCACCACC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr11_KI270832v1_alt,-24458,270M,0;"
3305,12142,16,chr11,501055,0,270M,*,0,0,GAGGTTGAGGCCGCATAAGCCATGAGGGAGCCACTCCAGCCTGGGT...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr11_KI270832v1_alt,-30696,270M,0;"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18209,23668,0,chr6_GL000256v2_alt,4869754,0,270M,*,0,0,CCCAGACTCTGACCCTCTTAAAGGCTCTTGACCAGTAAAATGTAAC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr6,+33421029,270M,0;"
18210,23670,0,chr6_GL000256v2_alt,4878480,0,270M,*,0,0,TGGGACATGGAGACCCAGGATCACAGTTTGTGCTTTTCAGGTTTGC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr6,+33429755,270M,0;"
18211,23677,0,chr6_GL000256v2_alt,4899195,0,270M,*,0,0,ATTAGTTTTGAGCCTGAATTTTAAAAAGCCAAGTGTTGCCCCCAGC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr6,+33450472,270M,0;"
18213,18450,16,chr19_GL949752v1_alt,161769,0,270M,*,0,0,CTTCTCCCCAGAGCTGGTTTTTCTCCTTTGCAAAATAGCTGGCTAC...,*,NM:i:0,MD:Z:270,AS:i:270,XS:i:270,"XA:Z:chr19_GL949746v1_alt,-161602,270M,0;chr19..."


In [135]:
matchable_header - matchable_header_subset
# found out it is because of an additional column

{10268,
 10269,
 12139,
 12140,
 12141,
 12142,
 12143,
 12144,
 12145,
 12146,
 12147,
 12148,
 12149,
 12150,
 12151,
 12152,
 12153,
 12154,
 12157,
 12159,
 12160,
 12164,
 12166,
 12168,
 12172,
 12173,
 12174,
 12175,
 12176,
 12177,
 12178,
 12179,
 12180,
 12181,
 12182,
 12183,
 12184,
 12189,
 12190,
 12191,
 12192,
 12193,
 12194,
 12195,
 12196,
 12211,
 12212,
 12215,
 12216,
 12221,
 12222,
 15330,
 15331,
 15456,
 15457,
 15458,
 15746,
 15747,
 15748,
 15749,
 15750,
 15751,
 15752,
 15753,
 15754,
 15755,
 15756,
 15757,
 15758,
 15759,
 15760,
 15761,
 15762,
 15763,
 15764,
 15765,
 15766,
 15770,
 15772,
 15775,
 15776,
 15778,
 15779,
 15780,
 17048,
 17049,
 17050,
 17051,
 17053,
 17054,
 17055,
 17056,
 17057,
 17058,
 17059,
 17060,
 17061,
 17062,
 17063,
 17064,
 17065,
 17066,
 17067,
 17068,
 17069,
 17070,
 17072,
 17073,
 17074,
 17075,
 17076,
 17077,
 17078,
 17079,
 17080,
 17081,
 17082,
 17083,
 17084,
 17085,
 17086,
 17087,
 17088,
 17089,
 17090,


### Check sequences duplicates: 
- lost sequences from first version of deduplication

In [ ]:
# example I noticed got lost:
tile1_1_mohlke = 'AGGACCGGATCAACTAGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCAGTGGCCTCTGCAGCAATTTATTATTCTTACATCAGATGTTTGAAGTAGGTGAAGGGGCAGGTGGCATGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAGTAGTGGTTGAAACAGAGCATTCAAAAAAGCGTACACTTTAACCTGTAATCCGCAAACATTCCTTTGAGTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGCATTGCGTGAACCGA'

In [28]:
design_file_df = hf.fasta_to_dataframe(config['files']['final_design']['first_design_fasta'])
design_file_df
initial_sequences = set(design_file_df['sequence'].to_list())
initial_header = set(design_file_df['header'].to_list())
print(len(initial_sequences))
print(len(initial_header))
print(design_file_df.shape[0])

80215
80084
80806


In [25]:
design_file_df.shape[0]

80806

In [31]:
design_file_df.loc[design_file_df['sequence'] == tile1_1_mohlke]['header'].to_list()
hf.write_fasta(design_file_df.loc[design_file_df['sequence'] == tile1_1_mohlke], '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/notebooks/testing_fasta.fa')

True

In [ ]:
design_no_duplicates_last_mohan = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates.fa'
design_no_duplicates_last_mohan_df = hf.fasta_to_dataframe(design_no_duplicates_last_mohan)
design_no_duplicates_last_mohan_df
last_mohan_sequences = set(design_no_duplicates_last_mohan_df['sequence'].to_list())
last_mohan_header = set(design_no_duplicates_last_mohan_df['header'].to_list())
print(len(last_mohan_sequences))
print(len(last_mohan_header))
print(design_no_duplicates_last_mohan_df.shape[0])
design_no_duplicates_last_mohan_df.loc[design_no_duplicates_last_mohan_df['sequence'] == tile1_1_mohlke]['header'].to_list()

80215
79493
80215


['GC_Selvarajan:REF_rs4846913|STARR-seq-HepG2_fwd_tile1-1']

In [30]:
from Bio import SeqIO
from collections import defaultdict

def merge_fasta_headers(input_fasta, output_fasta):
    # Dictionary to map sequences to headers
    sequence_to_headers = defaultdict(list)

    # Read the input fasta file
    for record in SeqIO.parse(input_fasta, "fasta"):
        sequence_to_headers[str(record.seq)].append(record.id)

    # Write the modified fasta to the output file
    with open(output_fasta, "w") as out_fasta:
        for seq, headers in sequence_to_headers.items():
            # Combine headers with '#'
            merged_header = "#".join(headers)
            # Write to the output fasta
            out_fasta.write(f">{merged_header}\n{seq}\n")

# Example usage
input_fasta = "testing_fasta.fa"
output_fasta = "output.fasta"
merge_fasta_headers(input_fasta, output_fasta)

In [26]:
second_version_design = hf.fasta_to_dataframe(config['files']['final_design']['second_design_fasta'])
second_version_design
second_sequences = set(second_version_design['sequence'].to_list())
print(len(second_sequences))

80215


In [27]:
second_version_design.shape[0]

80215

In [22]:
second_version_design.loc[second_version_design['sequence'] == tile1_1_mohlke]['header'].to_list()

['GC_Selvarajan:REF_rs4846913|STARR-seq-HepG2_fwd_tile1-1']

In [11]:
initial_sequences - second_sequences

set()

#### Removing duplicates: 

In [13]:
third_version_design = hf.fasta_to_dataframe(config['files']['final_design']['third_design_fasta_no_collisions'])
third_version_design
third_sequences = set(third_version_design['sequence'].to_list())
print(len(third_sequences))
initial_sequences - third_sequences

80141


{'...'}

In [18]:
fourth_design = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/removed_brackets_design_no_duplicates_sequence_and_header.fa'
fourth_design = '/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions_REF_to_elements.fa'
fourth_design = '/home/kisa/coding/80K_MPRA/design_data/removed_brackets_design_no_duplicates_sequence_and_header.fa'
fourth_version_design = hf.fasta_to_dataframe(fourth_design)
fourth_version_design
fourth_sequences = set(fourth_version_design['sequence'].to_list())
print(len(fourth_sequences))
initial_sequences - fourth_sequences

80215


set()

In [ ]:
fourth_version_design.loc[fourth_version_design['sequence'] == tile1_1_mohlke]['header'].to_list()

['GC_Selvarajan:REF_rs4846913|STARR-seq-HepG2_fwd_tile1-1']

['GC_Selvarajan:REF_rs4846913|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3']